**Act 3 to Act 4.** Act 3 mapped the MD-feature landscape: 41 per-target aggregates over ligand kinematics, protein stability, interaction fingerprints, and ligand chemistry. A few correlate weakly with is-active (`lig_buried_sasa_std_A2`, `lig_TPSA`). Most do not.

Act 4 asks the terminal question of the pilot: on the 9-target panel, can any of those features — used alone as a single-feature ranker, combined via an ML combo-selection pipeline, or rank-fused over top-K votes — beat GBSA-locked (0.541 [0.36, 0.71]) by more than its bootstrap CI half-width?

NB 22 sets the three reference bars (docking, GBSA, MD-composite). NBs 23–25 run the ML-combo direction (Claim A). NBs 26–27 cover physics-vs-empirical and the ligand-chem-only surrogate. NBs 28–29 test single-feature and rank-fusion (Claim B). NB 30 stratifies by target family. This is where the two headline verdicts land.

> **Reader guide.** *Experiment A1:* the reference three-bar panel BEDROC that every
> downstream cheap-configuration section compares against.
>
> **Question:** *on the discovery-9 panel, what are the reference BEDROC values for
> (i) docking alone, (ii) full-quality GBSA (reviewer-locked combo), and
> (iii) a naive MD-composite ranker?*
>
> **Method:** panel BEDROC α=20 with 95 % bootstrap CI (B=5000) over targets, computed
> per the two canonical conventions (9-target imputed / 8-target subset).
>
> **Reproducibility contract:** reads `data/derived/canonical_baselines.csv` (regenerated by
> `reproduce/canonical_baselines.py`); plot bar heights are directly from that CSV.

# 22 — BEDROC baselines

Sets the reference bars for Act 4: docking-only and GBSA-locked on the 9-target discovery panel. NBs 23, 24, and 25 use these two numbers as the "beat me" threshold.

See `docs/GLOSSARY.md` for term definitions.

In [ ]:
# --- notebook preamble ---
NB_STEM = "12_bedroc_baselines"

import sys, os, json, glob
from pathlib import Path

# Make the in-repo src package importable without an install
# find repo root robustly (walks up until pyproject.toml)
_repo_root = Path.cwd()
while _repo_root != _repo_root.parent and not (_repo_root / 'pyproject.toml').is_file():
    _repo_root = _repo_root.parent
sys.path.insert(0, str(_repo_root / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from discovery9.style import apply_style, NAVY, GOLD, GREY, GREY_DASH as GREYD, CREAM, WHITE, ACTIVE, DECOY, WARN
from discovery9.paths import ROOT, RAW, DERIVED, EXTERNAL, FIGURES, TABLES, GBSA_STUDY
from discovery9.io    import load_features, load_gbsa, load_gbsa_all, load_metadata, load_bedroc_matrix, load_bedroc_all_combos, load_per_complex_analysis
from discovery9.metrics import bedroc, bedroc_per_target, rank_fuse
apply_style()

# --- fig-capture hook (iter-3 fix) ---
_SAVED_FIGS = globals().setdefault('_SAVED_FIGS', [])
_orig_figure = plt.figure
_orig_subplots = plt.subplots
def _figure_capture(*a, **kw):
    fig = _orig_figure(*a, **kw)
    if fig not in _SAVED_FIGS:
        _SAVED_FIGS.append(fig)
    return fig
def _subplots_capture(*a, **kw):
    fig, ax = _orig_subplots(*a, **kw)
    if fig not in _SAVED_FIGS:
        _SAVED_FIGS.append(fig)
    return fig, ax
plt.figure = _figure_capture
plt.subplots = _subplots_capture

# Legacy monolith aliases:
ACTIVE_C, DECOY_C = ACTIVE, DECOY

# Default: load the master feature table (with ligand-chem descriptors when available)
df = load_features(with_ligand_chem=True)
print(f'features.parquet: {len(df)} complexes × {df.shape[1]} columns  ·  targets: {df.target.nunique()}')


## 1. Connect to BEDROC — join MD features with GBSA and docking scores

<!-- canonical baseline — see data/derived/canonical_baselines.csv -->

> **Panel note.** The bar chart below is on the historical 8-target subset (the join drops 4A5S). The canonical 9-target values (4A5S recovered) live in `data/derived/canonical_baselines.csv`:
>
> | Scorer | 8T (this NB) | Canonical 9T |
> |---|---:|---:|
> | Docking | 0.484 | **0.516 [0.30, 0.73]** |
> | GBSA-locked (`igb2_di4_salt0.15_st0.0072`) | 0.609 | **0.541 [0.36, 0.71]** (4A5S imputed at 0) |
> | MD-composite | 0.467 | ≈ 0.47 |
>
> We keep the 8T bars for continuity with earlier figures; the canonical CSV is the single source of truth for cross-notebook citations.

The upstream GBSA study gives per-complex MM-GBSA ΔG for one locked combo (`igb2_di4_salt0.15_st0.0072`). Our confirmatory metric is per-target BEDROC (Boltzmann-Enhanced Discrimination of ROC) at α=20 — heavily upweights the top of the ranked list. The join drops 4A5S, so the plotted numbers are the 8T subset; the canonical 9T values are in the callout above.

We join three sources on `(target, complex_id)`:

| Source | Columns brought in |
|---|---|
| `features.parquet` (this notebook) | all MD-derived features |
| `metadata.csv` (GBSA study) | `pchembl`, `docking_score` |
| `gbsa_dG_raw.csv` at locked combo | `mean_dG_kcalmol` |

That gives one row per complex with three candidate scorers (docking, GBSA, MD-ensemble) plus the ground-truth label (`is_active`). We then compute BEDROC α=20 per target for each scorer and compare.

In [ ]:

# NOTE(fix-pack iter1): removed hardcoded absolute path — GBSA_STUDY comes from discovery9.paths
LOCKED_COMBO = 'igb2_di4_salt0.15_st0.0072'

meta = pd.read_csv(f'{GBSA_STUDY}/data/raw/metadata.csv')
gbsa_raw = pd.read_csv(f'{GBSA_STUDY}/data/raw/gbsa_dG_raw.csv')
gbsa = gbsa_raw[gbsa_raw.combo == LOCKED_COMBO][['complex_id','target','mean_dG_kcalmol','n_frames']].rename(columns={'mean_dG_kcalmol':'gbsa_dG_kcalmol','n_frames':'gbsa_n_frames'})

print(f'metadata: {len(meta)} rows  ·  gbsa @ locked combo: {len(gbsa)} rows across {gbsa.target.nunique()} targets')

# Join: MD features ⊕ metadata (docking + pchembl) ⊕ gbsa
joined = (df
    .merge(meta[['complex_id','target','pchembl','docking_score']], on=['target','complex_id'], how='left')
    .merge(gbsa, on=['target','complex_id'], how='left'))
# drop the wrong-docking-box target explicitly
joined = joined[joined.target != '4A5S']
print(f'joined table: {len(joined)} rows  ·  targets: {sorted(joined.target.unique())}')
print(f'  complexes with docking score: {joined.docking_score.notna().sum()}')
print(f'  complexes with GBSA score   : {joined.gbsa_dG_kcalmol.notna().sum()}')
print(f'  complexes with is_active    : {joined.is_active.notna().sum()}')
JOINED = joined  # reuse in later cells

## 11. Per-target BEDROC — docking vs GBSA vs MD-features

BEDROC α=20 rewards early enrichment: ~80% of the weight lands on the first 8% of the ranked list. A more-negative raw score is better (stronger binder → lower energy → higher rank), so we flip signs where needed.

**Three scorers:**
- **`docking_score`** — negated. The pre-MD baseline.
- **`gbsa_dG_kcalmol`** — negated. Physics-based.
- **`md_stability_composite`** — z-score sum of `-lig_drift_mean_A`, `-lig_escape_frac`, `hb_persistence_frac`, `vdw_contacts_mean`, `ifp_tanimoto_median_vs_ref` (higher = stabler pose → active).

The MD composite is the cheapest signal we have; no energy calc needed. We compare it against docking (naive) and GBSA (physics) to see whether MD-stability alone can match the physics.

In [ ]:

def bedroc(scores, labels, alpha=20.0):
    '''Truchon-Bayly BEDROC. Higher scores rank first. Returns [0,1].'''
    scores = np.asarray(scores, dtype=float); labels = np.asarray(labels, dtype=int)
    m = np.isfinite(scores) & np.isfinite(labels)
    scores, labels = scores[m], labels[m]
    if labels.sum() == 0 or labels.sum() == len(labels):
        return np.nan
    order = np.argsort(-scores, kind='stable')
    labels = labels[order]
    N = len(labels); n = int(labels.sum()); Ra = n / N
    ranks = np.where(labels == 1)[0] + 1
    num = np.sum(np.exp(-alpha * ranks / N))
    denom = (Ra * (1 - np.exp(-alpha)) / (np.exp(alpha/N) - 1))
    Rf = num / denom if denom > 0 else np.nan
    factor = Ra * np.sinh(alpha/2) / (np.cosh(alpha/2) - np.cosh(alpha/2 - alpha*Ra))
    return Rf * factor + 1/(1 - np.exp(alpha*(1-Ra)))

# MD-stability composite: z-score within-target, sum of pro-stability contributions
MD_COMPOSITE_COLS = {
    'lig_drift_mean_A':          -1,   # smaller = better
    'lig_escape_frac':           -1,   # smaller = better
    'hb_persistence_frac':       +1,   # larger = better
    'vdw_contacts_mean':         +1,   # larger = better
    'ifp_tanimoto_median_vs_ref':+1,   # larger = better
    'lig_binding_modes_2A':      -1,   # fewer = better
}

def zscore(g): return (g - g.mean()) / (g.std(ddof=0) + 1e-9)

comp = pd.Series(0.0, index=JOINED.index)
for col, sign in MD_COMPOSITE_COLS.items():
    z = JOINED.groupby('target')[col].transform(zscore)
    comp = comp + sign * z.fillna(0)
JOINED['md_stability_composite'] = comp

rows = []
for tgt, g in JOINED.groupby('target'):
    lab = g.is_active.astype('boolean').astype('Int64').to_numpy()
    for scorer_name, series, direction in [
        ('docking',  -g.docking_score,          '↑'),
        ('gbsa',     -g.gbsa_dG_kcalmol,        '↑'),
        ('md_only',   g.md_stability_composite, '↑'),
    ]:
        b = bedroc(series, lab, alpha=20.0)
        rows.append({'target': tgt, 'scorer': scorer_name, 'bedroc_a20': b, 'n': int(g.is_active.notna().sum()), 'n_act': int((g.is_active == True).sum())})
bedroc_df = pd.DataFrame(rows)
bedroc_pivot = bedroc_df.pivot(index='target', columns='scorer', values='bedroc_a20').round(3)
print('BEDROC α=20 per target — higher is better (range [0,1]):')
bedroc_pivot

In [ ]:

fig, ax = plt.subplots(figsize=(11, 4.5))
x = np.arange(len(bedroc_pivot.index)); w = 0.27
scorers = [c for c in ['docking','gbsa','md_only'] if c in bedroc_pivot.columns]
colors = {'docking': GREYD, 'gbsa': NAVY, 'md_only': GOLD}
for i, s in enumerate(scorers):
    ax.bar(x + (i - len(scorers)/2 + 0.5)*w, bedroc_pivot[s].values, w,
           color=colors[s], edgecolor=NAVY, linewidth=0.6, label=s)
ax.set_xticks(x); ax.set_xticklabels(bedroc_pivot.index)
ax.axhline(0.1, color=GREY, ls=':', lw=1)
ax.set_ylabel('BEDROC α=20  (higher = better early enrichment)')
ax.set_title('Per-target BEDROC α=20  ·  docking (grey) vs GBSA (NAVY) vs MD-only composite (GOLD)')
ax.set_axisbelow(True); ax.yaxis.grid(True, color=GREY, alpha=0.5)
ax.legend(fontsize=9, loc='upper right')

**How to read it, per target:**
- **GBSA > docking** — the physics energy helps early enrichment. Usual outcome for a good pocket.
- **MD-only close to docking** — the cheap MD-stability composite already recovers some enrichment without an energy calc.
- **MD-only close to GBSA on some targets, worse on others** — the pockets where MD-stability tracks GBSA are the ones where an ML model on MD features could substitute the expensive GBSA rescoring.

If MD-only wins where GBSA loses, that points at complexes that are physically stable in MD but poorly scored by implicit-solvent GBSA — entropy-dominated binding, or metal-mediated contacts that igb2 mis-parameterises.

In [ ]:
# --- export every figure produced in this notebook (iter-3 fix) ---
try:
    FIGURES.mkdir(parents=True, exist_ok=True)
except NameError:
    from discovery9.paths import FIGURES
    FIGURES.mkdir(parents=True, exist_ok=True)
try:
    _cream = CREAM
except NameError:
    from discovery9.style import CREAM as _cream
figs = list(globals().get('_SAVED_FIGS', []))
# fallback: any figures still open in the backend
for num in plt.get_fignums():
    f = plt.figure(num)
    if f not in figs:
        figs.append(f)
saved = []
for i, fig in enumerate(figs, start=1):
    out = FIGURES / f"{NB_STEM}_fig{i}.png"
    try:
        fig.savefig(out, bbox_inches='tight', dpi=300, facecolor=_cream)
    except Exception as e:
        print(f'  WARN: failed to save fig{i}: {e}')
        continue
    saved.append(str(out.name))
print(f'saved {len(saved)} figures:')
for s in saved:
    print(' ', s)
